In [1]:
import pandas as pd

### Load the dataset

In [2]:
df = pd.read_csv("../Dataset/final.csv")
df.head()

,acc_x_mean,acc_x_std,acc_x_min,acc_x_max,acc_y_mean,acc_y_std,acc_y_min,acc_y_max,acc_z_mean,acc_z_std,...,net_acc_mean,net_acc_std,net_acc_min,net_acc_max,ecg_peak_freq,resp_peak_freq,temp_slope,eda_slope,label,subject
0,0.872800,0.024465,0.5070,1.1758,-0.132889,0.031347,-0.4014,0.1010,-0.284486,0.047498,...,0.929273,0.025727,0.522995,1.506039,2.200000,0.433333,-0.000012,-0.000033,0,2
1,0.875788,0.059040,0.4826,1.2712,-0.120982,0.078589,-0.4616,0.1268,-0.245599,0.127737,...,0.929947,0.056012,0.496726,1.303793,2.600000,0.066667,-0.000037,-0.000015,0,2
2,0.861583,0.021205,0.7454,1.1624,-0.156330,0.020616,-0.2042,-0.1002,-0.316530,0.031280,...,0.931918,0.018373,0.785213,1.185841,1.300000,0.100000,0.000004,0.000032,0,2
3,0.862690,0.009803,0.7906,0.9224,-0.181321,0.022755,-0.2394,-0.1370,-0.306185,0.014533,...,0.933610,0.007607,0.882272,0.981359,2.466667,0.366667,0.000003,-0.000026,0,2
4,0.857272,0.006321,0.8050,0.9246,-0.188318,0.022537,-0.2520,-0.1382,-0.320673,0.011200,...,0.934800,0.005797,0.882175,0.996625,1.200000,0.400000,-0.000011,-0.000036,0,2


In [3]:
df.columns

Index(['acc_x_mean', 'acc_x_std', 'acc_x_min', 'acc_x_max', 'acc_y_mean',
       'acc_y_std', 'acc_y_min', 'acc_y_max', 'acc_z_mean', 'acc_z_std',
       'acc_z_min', 'acc_z_max', 'ecg_mean', 'ecg_std', 'ecg_min', 'ecg_max',
       'emg_mean', 'emg_std', 'emg_min', 'emg_max', 'eda_mean', 'eda_std',
       'eda_min', 'eda_max', 'temp_mean', 'temp_std', 'temp_min', 'temp_max',
       'resp_mean', 'resp_std', 'resp_min', 'resp_max', 'net_acc_mean',
       'net_acc_std', 'net_acc_min', 'net_acc_max', 'ecg_peak_freq',
       'resp_peak_freq', 'temp_slope', 'eda_slope', 'label', 'subject'],
      dtype='str')

### Drop the subject column

In [14]:
df.drop(columns=['subject'], inplace=True)

### check for null values

In [6]:
count = 0
for numCount in df.isnull().sum():
    if(numCount > 0):
        count += 1
print("Number of columns with null values: " + str(count))

Number of columns with null values: 0


### Check for duplicate

In [7]:
df.duplicated().sum()

np.int64(0)

### Categories the dataset

In [8]:
df["label"].value_counts()

label
0    1310
1     581
4     387
2     326
3     180
7      19
6      18
5      18
Name: count, dtype: int64

according to dataset
- 0 = not defined / transient
- 1 = baseline (not stress)
- 2 = stress
- 3 = amusement (not stress)
- 4 = meditation
- 5/6/7 = should be ignored in this dataset

In [9]:
accept_label = [1, 2, 3, 4]
df = df[df["label"].isin(accept_label)]
df["label"].value_counts()

label
1    581
4    387
2    326
3    180
Name: count, dtype: int64

In [10]:
nStr = [1, 3, 4]
def apply_target(label):
    if label == 2:
        return 1
    elif label in nStr:
        return 0
    else:
        return "unknown"

df["label"] = df["label"].apply(apply_target)
df["label"].value_counts()

label
0    1148
1     326
Name: count, dtype: int64

### Train test split

In [15]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, random_state=42)

In [16]:
x_train = train.drop("label", axis=1)
y_train = train["label"]

y_test = test["label"]
x_test = test.drop("label", axis=1)

In [17]:
x_train.shape

(1179, 40)

In [21]:
y_train.value_counts()

label
0    925
1    254
Name: count, dtype: int64

Oversampling

In [24]:
from imblearn.over_sampling import SMOTE

smote = SMOTE()
X_res, y_res = smote.fit_resample(x_train, y_train)

In [25]:
y_res.value_counts()

label
0    925
1    925
Name: count, dtype: int64

### Save the train and test dataset

In [ ]:
# dataset will be used for training the model
X_res.to_csv("../Dataset/train_feature.csv", index=False)
y_res.to_csv("../Dataset/train_target.csv", index=False)

# dataset will be used for testing / validating the model
x_test.to_csv("../Dataset/val_feature.csv", index=False)
y_test.to_csv("../Dataset/val_target.csv", index=False)
print("Saved")

Saved
